# 🛡️ Insurance Policy Status Classifier
### End-to-End ML Pipeline — Decision Tree · Random Forest · Gradient Boosted Tree
---

## Step 1 — Import Required Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('✅ All packages imported successfully.')

**Explanation:**
- `pandas` / `numpy` — data manipulation and numerical operations
- `matplotlib` / `seaborn` — visualisations
- `sklearn` — Label encoding, train-test split, classifiers, metrics

---
## Step 2 — Basic Data Check & Remove Unwanted Columns

In [ ]:
# Load dataset
df = pd.read_csv('Insurance__1_.csv')

print('=== Shape ===')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Null Value Count ===')
print(df.isnull().sum())

print('\n=== Null Percentage ===')
print((df.isnull().mean() * 100).round(2))

print('\n=== First 5 Rows ===')
df.head()

In [ ]:
# Remove columns not needed for modelling
cols_to_drop = ['POLICY_NO', 'PI_NAME']
df.drop(columns=cols_to_drop, inplace=True)

print(f'✅ Dropped columns: {cols_to_drop}')
print(f'Remaining shape: {df.shape}')
print(df.columns.tolist())

**Explanation:**
- `POLICY_NO` is a unique identifier — carries no predictive information.
- `PI_NAME` is a free-text name — not useful and can cause data leakage.
- We inspect nulls, dtypes, and shape before any transformation.

---
## Step 3 — Handle Missing Values

In [ ]:
# Fix comma-formatted numeric columns first
for col in ['SUM_ASSURED', 'PI_ANNUAL_INCOME']:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '').str.strip(),
            errors='coerce'
        )

# Identify column types
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Exclude label from imputation
TARGET = 'POLICY_STATUS'
if TARGET in categorical_cols:
    categorical_cols.remove(TARGET)

print('Numeric columns   :', numeric_cols)
print('Categorical cols  :', categorical_cols)

# Impute numeric with MEAN
for col in numeric_cols:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        fill = df[col].mean()
        df[col].fillna(fill, inplace=True)
        print(f'  [NUMERIC] {col}: filled {n_null} nulls with mean={fill:.4f}')

# Impute categorical with MODE (most frequent)
for col in categorical_cols:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        fill = df[col].mode()[0]
        df[col].fillna(fill, inplace=True)
        print(f'  [CATEG]   {col}: filled {n_null} nulls with mode="{fill}"')

print(f'\n✅ Remaining nulls: {df.isnull().sum().sum()}')

**Explanation:**
- **Numeric columns** → filled with the column **mean** (preserves overall distribution).
- **Categorical columns** → filled with the **mode** (most frequent category).
- `SUM_ASSURED` and `PI_ANNUAL_INCOME` had comma-formatted strings — cleaned before imputation.

---
## Step 4 — Label Encoding

In [ ]:
df_encoded  = df.copy()
le_dict     = {}     # store encoders for later inverse-transform
mapping_log = []

for col in df_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    le_dict[col] = le
    for orig, enc in zip(le.classes_, le.transform(le.classes_)):
        mapping_log.append({'Column': col, 'Original': orig, 'Encoded': int(enc)})

mapping_df = pd.DataFrame(mapping_log)
print('=== Encoding Mapping (first 20 rows) ===')
print(mapping_df.head(20).to_string(index=False))

# Save mapping to CSV
mapping_df.to_csv('encoding_map.csv', index=False)
print('\n✅ Encoding map saved → encoding_map.csv')

print('\n=== Encoded DataFrame (first 5 rows) ===')
df_encoded.head()

**Explanation:**
- `LabelEncoder` converts each unique string to an integer (0-based, alphabetical order).
- `le_dict` stores every encoder so we can reverse-map predictions back to original labels.
- The full mapping is saved as **encoding_map.csv** for audit / reference.

---
## Step 5 — Split Data and Label

In [ ]:
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f'Feature matrix X : {X.shape}')
print(f'Label vector   y : {y.shape}')
print(f'\nFeatures used:\n{X.columns.tolist()}')
print(f'\nLabel distribution:')
label_counts = y.value_counts().reset_index()
label_counts.columns = ['Encoded', 'Count']
label_counts['Original'] = le_dict[TARGET].inverse_transform(label_counts['Encoded'])
print(label_counts.to_string(index=False))

**Explanation:**
- `X` contains all feature columns (everything except `POLICY_STATUS`).
- `y` contains the encoded target column.
- We display the class distribution to check for class imbalance.

---
## Step 6 — Train / Test Split (80:20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # preserves class ratio in both splits
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'\nClass distribution in train set:\n{y_train.value_counts()}')
print(f'\nClass distribution in test  set:\n{y_test.value_counts()}')

**Explanation:**
- `test_size=0.20` → 80% train, 20% test.
- `stratify=y` ensures both sets have the same class proportion as the full dataset.
- `random_state=42` for reproducibility.

---
## Step 7 — Train Classification Models

In [ ]:
# ── Decision Tree ──────────────────────────────────────────────────────────
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
print('✅ Decision Tree trained')

# ── Random Forest ──────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print('✅ Random Forest trained')

# ── Gradient Boosted Tree ──────────────────────────────────────────────────
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)
print('✅ Gradient Boosted Tree trained')

**Explanation:**
- **Decision Tree** — single tree, `max_depth=5` prevents overfitting.
- **Random Forest** — ensemble of 100 trees; reduces variance via bagging.
- **Gradient Boosted Tree** — sequential ensemble; reduces bias via boosting.

---
## Step 8 — Training & Testing Accuracy (Tabulated)

In [ ]:
models = {
    'Decision Tree':         dt_model,
    'Random Forest':         rf_model,
    'Gradient Boosted Tree': gb_model,
}

results = {}
acc_rows = []

for name, model in models.items():
    tr_acc = accuracy_score(y_train, model.predict(X_train)) * 100
    te_acc = accuracy_score(y_test,  model.predict(X_test))  * 100
    results[name] = {'model': model, 'train': tr_acc, 'test': te_acc,
                     'y_pred': model.predict(X_test)}
    acc_rows.append({'Model': name,
                     'Train Accuracy (%)': round(tr_acc, 2),
                     'Test Accuracy (%)':  round(te_acc, 2),
                     'Delta (Train-Test)': round(tr_acc - te_acc, 2)})

acc_df = pd.DataFrame(acc_rows)
print(acc_df.to_string(index=False))

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(9, 4))
x  = np.arange(len(acc_df))
w  = 0.35
ax.bar(x - w/2, acc_df['Train Accuracy (%)'], w, label='Train', color='#1a237e', alpha=0.85)
ax.bar(x + w/2, acc_df['Test Accuracy (%)'],  w, label='Test',  color='#3949ab', alpha=0.70)
ax.set_xticks(x)
ax.set_xticklabels(acc_df['Model'], fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training vs Testing Accuracy', fontweight='bold')
ax.set_ylim(50, 105)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Chart saved → accuracy_comparison.png')

**Explanation:**
- The table shows both train and test accuracy, plus the gap (Δ) indicating overfitting.
- A grouped bar chart provides a visual comparison across all models.

---
## Step 9 — Confusion Matrices

In [ ]:
class_names = le_dict[TARGET].classes_.tolist()
cmap = sns.light_palette('#1a237e', as_cmap=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('white')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])

    # Label TP / FP / TN / FN for binary classification
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        annot = np.array([[f'TN\n{tn}', f'FP\n{fp}'],
                           [f'FN\n{fn}', f'TP\n{tp}']])
    else:
        annot = cm.astype(str)

    sns.heatmap(cm, annot=annot, fmt='', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=.5, ax=ax, cbar=False, annot_kws={'size': 9})
    ax.set_title(name, fontweight='bold', color='#1a237e', pad=8)
    ax.set_xlabel('Predicted Label', fontsize=8)
    ax.set_ylabel('True Label',      fontsize=8)
    ax.tick_params(labelsize=7, rotation=15)

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → confusion_matrices.png')

# Print classification report for each model
for name, res in results.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_test, res['y_pred'], target_names=class_names))

**Explanation:**
- **TP** (True Positive) — correctly predicted positive class.
- **TN** (True Negative) — correctly predicted negative class.
- **FP** (False Positive) — predicted positive but actually negative (Type I error).
- **FN** (False Negative) — predicted negative but actually positive (Type II error).
- `classification_report` adds Precision, Recall and F1-Score per class.

---
## Step 10 — Feature Importance Charts

In [ ]:
feat_names = X.columns.tolist()
top_n      = min(15, len(feat_names))
colors     = ['#1a237e', '#3949ab', '#283593']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

for ax, (name, res), color in zip(axes, results.items(), colors):
    imps = res['model'].feature_importances_
    idx  = np.argsort(imps)[::-1][:top_n]
    top_f = [feat_names[i] for i in idx]
    top_v = imps[idx]

    ax.barh(range(top_n), top_v[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_f[::-1], fontsize=8)
    ax.set_xlabel('Importance Score', fontsize=8)
    ax.set_title(name, fontweight='bold', color='#1a237e', pad=8)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('Feature Importance — All Models', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → feature_importance.png')

**Explanation:**
- `feature_importances_` attribute is built-in for all tree-based sklearn models.
- Higher value = greater contribution to reducing impurity (Gini / MSE).
- Comparing across all three models reveals which features are consistently important.